# Which modulation band's edges drive the response?

Notebook 7 sweeps a **low-pass** cutoff on the onset feature and asks how sharp an edge has to be
to count. This asks the other half of the question: an envelope rises on many timescales at
once, and a rise in the 2-4 Hz band is a different acoustic event from a rise in the
100-200 Hz band. Band-limiting before differentiating separates them:

$$x_{[a,b]}(t) = \max\!\left(\frac{\mathrm{d}}{\mathrm{d}t}\,\mathrm{bandpass}_{[a,b]}\big[x_{\text{env}}(t)\big],\; 0\right)$$

`scripts/trfs/get_onset_grid.py` fits a transfer function for every cell of a grid of
$(a, b)$ and pools it over participants. The bottom row of that grid, $a = 0$, is exactly
notebook 7's cutoff sweep, so the two analyses are nested rather than merely similar.

**The interpretation problem is the whole problem.** A band's $|H|$ is not comparable with
another band's as it stands, for two reasons that have nothing to do with the brain:

1. $H = S_{xy}/S_{xx}$ is homogeneous of degree $-1$ in the feature, and a narrow high band's
   feature is orders of magnitude smaller than a wide low one's. Multiplying the feature's own
   SD back in removes it.
2. Even then, a band with no response at all returns noise, not zero -- and how much noise
   depends on how many independent samples the band affords.

So the notebook reports four readings of every cell, and they answer different questions:

| | what it asks |
|---|---|
| **peak amplitude** | how large, in the EEG's own units |
| **standardised amplitude** | how large per unit of regressor |
| **baseline SNR** | how large against this band's own pre-stimulus noise |
| **split-half $r$** | how *reproducible*, across two halves of the participants |
| **null SNR** | how large against the same band fitted on mismatched stimulus/EEG pairs |

The last two are the ones that can distinguish a response from a well-scaled artefact.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm, TwoSlopeNorm
from mne import pick_info
from mne.io import read_raw_fif
from mne.viz import plot_topomap
from mne.viz.evoked import _get_pos_outlines, _rgb
from mne.viz.topomap import _draw_outlines
from scipy.signal import firwin, freqz, hilbert, iirnotch, kaiserord

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())
sys.path.insert(0, str(ROOT / "src"))

from trfs.onset_grid import resolve_onset_grid_dir

# --- what is read ------------------------------------------------------------
REFERENCE = "avg"
WINDOW_SECONDS, OVERLAP = 5.0, 0.8

GRID_DIR = resolve_onset_grid_dir(CONFIG, REFERENCE, WINDOW_SECONDS, OVERLAP, ROOT)
GRID_PATH = GRID_DIR / "onset_grid.npz"
NULL_PATH = GRID_DIR / "onset_grid_null.npz"
EEG_DIR = Path(CONFIG["eeg_dir"])
OUT_DIR = ROOT / "figures"

if not GRID_PATH.exists():
    raise FileNotFoundError(
        f"{GRID_PATH} is missing.\nRun: python scripts/trfs/get_onset_grid.py")

with np.load(GRID_PATH) as saved:
    # (2, n_bands, F, n_channels): the two halves of the participants, pooled separately.
    GRID_FITS = saved["fits"]
    BAND_LOW, BAND_HIGH = saved["band_low"], saved["band_high"]
    EDGES = saved["edges"]
    frequencies = saved["frequencies"]
    channels = [str(name) for name in saved["channel_names"]]
    FEATURE_SD = saved["feature_sd"]
    EEG_SD = saved["eeg_sd"]
    fs = float(saved["fs"])
    nperseg = int(saved["nperseg"])
    n_trials = int(saved["n_trials"])
    listening_hours = float(saved["hours"])
    half_sizes = saved["half_sizes"]

HAS_NULL = NULL_PATH.exists()
if HAS_NULL:
    with np.load(NULL_PATH) as saved:
        NULL_FITS = saved["fits"]
        assert np.array_equal(saved["band_low"], BAND_LOW), "null grid has different bands"

# Zero lag in the middle of the transform, which is circular with the window's period.
times = (np.arange(nperseg) - nperseg // 2) / fs * 1000

TEMPLATE_EEG = EEG_DIR / "sub-001_-_ses-001_-_trial-001_-_cond-natural_-_audio01-audiobook_5_1_-_eeg.fif"
RAW_INFO = read_raw_fif(TEMPLATE_EEG, preload=False, verbose="ERROR").pick("eeg").info
INFO = pick_info(RAW_INFO, [RAW_INFO.ch_names.index(name) for name in channels])
CHANNEL_COLOURS = _rgb(*np.array([channel["loc"][:3] for channel in INFO["chs"]]).T)

# --- house style, shared with figures 1-7 ------------------------------------
DEFAULT_FONTSIZE, TICK_FONTSIZE, SMALL_FONTSIZE = 10, 8, 6.5
PANEL_LABEL_FONTSIZE = 14
INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE, "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE, "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8, "figure.dpi": 130,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def bare(ax):
    ax.spines[["top", "right"]].set_visible(False)
    return ax


def plot_sensor_key(ax):
    positions, outlines = _get_pos_outlines(INFO, np.arange(len(INFO.ch_names)),
                                            sphere=(0, 0, 0, 0.095))
    _draw_outlines(ax, outlines)
    ax.scatter(*positions.T, c=CHANNEL_COLOURS, s=7, linewidths=0)
    ax.set_aspect("equal")
    ax.set_axis_off()


print(f"{GRID_PATH.name}: {len(BAND_LOW)} bands from {len(EDGES)} edges "
      f"({EDGES[0]:g}-{EDGES[-1]:g} Hz)")
print(f"  {len(frequencies)} EEG bins to {frequencies[-1]:g} Hz, {len(channels)} channels")
print(f"  {n_trials} trials, {half_sizes[0]} + {half_sizes[1]} participants, "
      f"{listening_hours:.1f} h")
print(f"  null: {'present' if HAS_NULL else 'MISSING -- run with --null for the SNR floor'}")

## Reading a band

The pooled file holds $H$ on the EEG frequency axis, so a band's transfer function is one
multiplication along frequency and an inverse transform -- the display band is a choice made
here, not in the run.

The same Kaiser design as notebooks 6 and 7, with the **magnitude** applied rather than the
complex response so it is zero phase: these are read for their latency, and a group delay would
bias exactly that.

In [ ]:
# --- options -----------------------------------------------------------------
DISPLAY_BAND = (0.2, 15)      # the EEG band every cell is read through
FILTER_TRANSITION_HZ = 1.0
FILTER_RIPPLE_DB = 60.0
MAINS_HZ, NOTCH_BANDWIDTH_HZ = 50.0, 2.0

# How a cell's transfer function is collapsed over the channel axis before it is read.
# "rms" is the whole montage -- the reduction every other notebook here uses, and the one
# that picks no cluster by hand. At 134.7 h the channels away from the response add their
# own noise floor rather than obscuring the response, and the floors it is divided by are
# measured through the same reduction, so what it costs in contrast it returns in not
# having chosen where to look. "cluster" is the fronto-central mean below.
CHANNEL_REDUCTION = "rms"     # "rms" over every channel | "cluster" mean

# The response to an acoustic onset is fronto-central, so this is where "cluster" measures.
GRID_CLUSTER = ["F1", "Fz", "F2", "FC1", "FCz", "FC2", "C1", "Cz", "C2"]
# =============================================================================

cluster = [INFO.ch_names.index(name) for name in GRID_CLUSTER if name in INFO.ch_names]
assert cluster, "none of GRID_CLUSTER is in this reference's montage"


def notch_response():
    """Every mains harmonic inside the kept range, as a product of `iirnotch` magnitudes."""
    response = np.ones_like(frequencies)
    for harmonic in np.arange(MAINS_HZ, frequencies[-1] + MAINS_HZ, MAINS_HZ):
        if harmonic >= fs / 2:
            break
        b, a = iirnotch(harmonic, harmonic / NOTCH_BANDWIDTH_HZ, fs=fs)
        response *= np.abs(freqz(b, a, worN=frequencies, fs=fs)[1])
    return response


NOTCHES = notch_response()
_responses: dict[tuple, np.ndarray] = {}


def display_response(low, high, transition=FILTER_TRANSITION_HZ):
    """The display filter's magnitude on the kept frequency axis, cached."""
    key = (float(low), float(high), float(transition))
    if key not in _responses:
        taps, beta = kaiserord(FILTER_RIPPLE_DB, transition / (fs / 2))
        taps = min(taps, nperseg - 1) | 1
        design = firwin(taps, [low, high], window=("kaiser", beta), pass_zero=False, fs=fs)
        _responses[key] = np.abs(freqz(design, worN=frequencies, fs=fs)[1]) * NOTCHES
    return _responses[key]


def band_trf(index, fits=None, band=None):
    """One grid cell in the lag domain: `(n_lags, n_channels)`.

    Zero-padded back to the full transform length, so the lag axis is the same 0.98 ms grid the
    other notebooks use even though only the low end of the spectrum was kept.
    """
    source = GRID_FITS if fits is None else fits
    spectrum = np.zeros((nperseg // 2 + 1, source.shape[-1]), complex)
    spectrum[:len(frequencies)] = (source[:, index].mean(0)
                                   * display_response(*(band or DISPLAY_BAND))[:, None])
    return np.roll(np.fft.irfft(spectrum, n=nperseg, axis=0), nperseg // 2, axis=0)


def reduce_channels(trf):
    """Collapse the channel axis of a transfer function, however the option asks.

    The RMS is the instantaneous spread of the field over the scalp, so it is positive
    and still oscillates with the response -- a global field power. The cluster mean is
    signed. What a peak means differs between them, which is why `response_peak` below
    is the only place either is searched for one.
    """
    if CHANNEL_REDUCTION == "rms":
        return np.sqrt((trf ** 2).mean(-1))
    return trf[..., cluster].mean(-1)


def response_peak(trace, window):
    """The largest value of a reduced trace inside `window`, and where it falls.

    Under "rms" the trace is already non-negative and its maximum is read straight off.
    Under "cluster" it is a signed waveform, and the analytic envelope is taken first so
    that a trough counts as much as a crest and the value does not depend on which half
    of a cycle the response happens to land in. Hilbert is not applied to the RMS: it
    assumes a zero-mean narrowband signal, and the RMS is neither.
    """
    reading = trace if CHANNEL_REDUCTION == "rms" else np.abs(hilbert(trace))
    inside = reading[window]
    return inside.max(), times[window][np.argmax(inside)]


def band_reduced(index, fits=None, band=None):
    """One cell's transfer function, collapsed over the channels."""
    return reduce_channels(band_trf(index, fits, band))


print(f"display band {DISPLAY_BAND[0]:g}-{DISPLAY_BAND[1]:g} Hz; channels: "
      + ("RMS over all " + str(len(INFO.ch_names))
         if CHANNEL_REDUCTION == "rms"
         else "mean of " + ", ".join(INFO.ch_names[i] for i in cluster)))

## Five readings of every cell

Computed once, for every band, and used by all the figures below.

* **peak** -- the largest analytic-envelope value of the fronto-central mean inside the response
  window, in the EEG's own units. What you would quote as an amplitude.
* **standardised** -- the same times the feature's own SD. $H$ is homogeneous of degree $-1$ in
  the regressor, so without this a narrow band looks enormous purely because its feature is
  tiny.
* **baseline SNR** -- peak over the RMS of the same trace in a pre-stimulus window. Cheap, and
  it needs no extra run, but note what it assumes: that negative lags are noise. The transform
  is circular and the display filter rings, so a window too close to zero is contaminated by the
  response itself.
* **split-half $r$** -- the correlation between the two halves of the participants over the
  response window. This is the one that cannot be faked by a scaling: a band whose response is
  noise has nothing for the halves to agree on.
* **null SNR** -- peak over the RMS of the same band fitted on **mismatched** stimulus/EEG
  pairs, taken over a wide window around zero. Real data on both sides, wrong pairing, so it
  carries every bias the estimator has except a real response. The floor is an RMS and not that
  trace's own peak: the maximum of a noise trace rests on one sample and grows with the number
  of lags it is searched over, so it moves with the window as much as with the data.

The two floors are formed the same way -- an envelope peak over a trace RMS -- so they sit on
one scale, and they answer the same question two ways. They can disagree: the baseline assumes
negative lags are noise and is contaminated by filter ringing at short lags, the null is not,
but the null costs a second run of the whole grid.

In [ ]:
# --- options -----------------------------------------------------------------
RESPONSE_WINDOW = (0.2, 20.0)      # ms, where a response is looked for
BASELINE_WINDOW = (-1000.0, -500.0)  # ms, taken as noise
# Where the null's own scale is measured. An RMS over a wide window around zero rather
# than the peak inside the response window: the peak of a noise trace rests on a single
# sample and grows with the number of lags it is taken over, so it is a normaliser that
# moves with the window as much as with the data. An RMS over ~1000 lags is the same
# floor estimated stably. Spanning zero is deliberate -- a mispaired fit has no response
# in it, so these are the lags the comparison is actually made at, ringing and all.
NULL_WINDOW = (-500.0, 500.0)      # ms, the null's own scale
# =============================================================================

response = (times >= RESPONSE_WINDOW[0]) & (times <= RESPONSE_WINDOW[1])
baseline = (times >= BASELINE_WINDOW[0]) & (times <= BASELINE_WINDOW[1])
null_window = (times >= NULL_WINDOW[0]) & (times <= NULL_WINDOW[1])
assert baseline.sum() > 50, "baseline window is too short to estimate an RMS"

n_bands = len(BAND_LOW)
peak = np.zeros(n_bands)
peak_lag = np.zeros(n_bands)
baseline_rms = np.zeros(n_bands)
split_half = np.zeros(n_bands)
null_rms = np.full(n_bands, np.nan)

for index in range(n_bands):
    halves = np.stack([
        np.fft.irfft(
            np.pad(GRID_FITS[half, index] * display_response(*DISPLAY_BAND)[:, None],
                   ((0, nperseg // 2 + 1 - len(frequencies)), (0, 0))),
            n=nperseg, axis=0)
        for half in (0, 1)])
    halves = np.roll(halves, nperseg // 2, axis=1)          # (2, n_lags, n_channels)

    # The pooled response is the two halves averaged and then reduced, not the two
    # reduced traces averaged: under "rms" those are different, and the first is the
    # field of the whole group rather than the mean of two groups' field strengths.
    trace = reduce_channels(halves.mean(0))
    peak[index], peak_lag[index] = response_peak(trace, response)
    baseline_rms[index] = np.sqrt((trace[baseline] ** 2).mean())

    # Over the response window only: the halves agree trivially on a flat baseline, and
    # including it would inflate every cell towards 1. Under "rms" the correlation is
    # taken over lags *and* channels on the signed waveforms, so it asks the halves to
    # agree on the topography as well as the time course; two RMS traces would agree in
    # part simply by both being positive.
    if CHANNEL_REDUCTION == "rms":
        split_half[index] = np.corrcoef(halves[0][response].ravel(),
                                        halves[1][response].ravel())[0, 1]
    else:
        reduced = reduce_channels(halves)
        split_half[index] = np.corrcoef(reduced[0][response], reduced[1][response])[0, 1]

    if HAS_NULL:
        null_trace = band_reduced(index, NULL_FITS)
        null_rms[index] = np.sqrt((null_trace[null_window] ** 2).mean())

standardised = peak * FEATURE_SD
baseline_snr = peak / baseline_rms
# Envelope peak over trace RMS, exactly as `baseline_snr` is formed, so the two floors
# are read on the same scale and their disagreement is about the floor and not about
# how each was measured.
null_snr = peak / null_rms if HAS_NULL else np.full(n_bands, np.nan)

METRICS = {
    "peak": (peak, "Peak envelope (a.u.)"),
    "standardised": (standardised, r"Peak $\times$ feature SD"),
    "baseline_snr": (baseline_snr, "Peak / baseline RMS"),
    "split_half": (split_half, "Split-half $r$"),
}
if HAS_NULL:
    METRICS["null_snr"] = (null_snr, "Peak / null RMS")

print(f"{n_bands} bands, response {RESPONSE_WINDOW[0]:g}-{RESPONSE_WINDOW[1]:g} ms, "
      f"baseline {BASELINE_WINDOW[0]:g}-{BASELINE_WINDOW[1]:g} ms, "
      f"null {NULL_WINDOW[0]:g}-{NULL_WINDOW[1]:g} ms")
print(f"\n{'metric':>14s} {'min':>10s} {'median':>10s} {'max':>10s}   best band")
for name, (values, _) in METRICS.items():
    best = int(np.nanargmax(values))
    print(f"{name:>14s} {np.nanmin(values):10.3g} {np.nanmedian(values):10.3g} "
          f"{np.nanmax(values):10.3g}   {BAND_LOW[best]:g}-{BAND_HIGH[best]:g} Hz "
          f"@ {peak_lag[best]:.0f} ms")

## The grid

Each cell is a band, placed at its low edge on x and its high edge on y, so the diagonal is the
narrowest bands and the top-left corner the widest. The edges are unevenly spaced -- 1 Hz to
30 Hz, then 5 Hz -- and the axes are drawn on the real frequencies rather than on cell indices,
so the low-frequency corner is not given more visual weight than it has evidence for.

Read the four panels together. Amplitude alone will favour whatever band has the largest
regressor; split-half $r$ alone is insensitive to size. A band that is high on **both** is a
real, reproducible response.

In [ ]:
# --- options -----------------------------------------------------------------
GRID_FIGSIZE = (6.27, 6.4)
GRID_PANELS = ["standardised", "baseline_snr", "split_half"] + (["null_snr"] if HAS_NULL
                                                                else ["peak"])
GRID_CMAP = "magma"
GRID_MAX_EDGE = None            # None shows the whole grid; a number zooms to that edge
# =============================================================================


def grid_matrix(values):
    """The flat band axis as an (n_edges, n_edges) upper-triangular matrix, NaN elsewhere."""
    lookup = {edge: index for index, edge in enumerate(EDGES)}
    matrix = np.full((len(EDGES), len(EDGES)), np.nan)
    for value, low, high in zip(values, BAND_LOW, BAND_HIGH):
        matrix[lookup[high], lookup[low]] = value
    return matrix


# Cell boundaries from the edge list, so an unevenly spaced grid is drawn to scale. The last
# boundary is extrapolated from the final step.
steps = np.diff(EDGES)
bounds = np.concatenate([EDGES - np.concatenate([[steps[0]], steps]) / 2,
                         [EDGES[-1] + steps[-1] / 2]])

grid_fig, grid_axes = plt.subplots(2, 2, figsize=GRID_FIGSIZE)
for ax, name in zip(grid_axes.ravel(), GRID_PANELS):
    values, label = METRICS[name]
    matrix = grid_matrix(values)
    # Every panel on its own 2nd-98th percentile, split-half r included. A correlation would
    # normally be shown on a fixed -1..1, but measured it runs 0.89 to 0.997 here -- each half
    # is already 40 participants and 67 hours, so every band's response is reproducible and a
    # full-range scale renders the panel a single flat colour. The null is what says that is
    # real rather than an artefact of the metric: the same correlation on mismatched pairs has
    # a median of 0.06.
    finite = matrix[np.isfinite(matrix)]
    limits = dict(vmin=np.percentile(finite, 2), vmax=np.percentile(finite, 98),
                  cmap=GRID_CMAP)
    mesh = ax.pcolormesh(bounds, bounds, matrix, shading="flat", rasterized=True, **limits)
    ax.plot(bounds[[0, -1]], bounds[[0, -1]], color=HAIRLINE, lw=0.6, zorder=3)
    ax.set_xlabel("Low edge (Hz)")
    ax.set_ylabel("High edge (Hz)")
    ax.set_title(label, fontsize=DEFAULT_FONTSIZE)
    ax.set_xlim(0, 100)
    if GRID_MAX_EDGE is not None:
        ax.set_xlim(bounds[0], GRID_MAX_EDGE)
        ax.set_ylim(bounds[0], GRID_MAX_EDGE)
    colorbar = plt.colorbar(mesh, ax=ax, pad=0.02)
    colorbar.ax.tick_params(labelsize=TICK_FONTSIZE)

for ax, letter in zip(grid_axes.ravel(), "abcd"):
    ax.text(-0.22, 1.04, f"{letter})", transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")
grid_fig.tight_layout()

stem = f"onset_grid_{REFERENCE}"
grid_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
grid_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

# The bands each metric likes best, which is the summary the grid is for.
print(f"{'metric':>14s}   top 5 bands")
for name in GRID_PANELS:
    values = METRICS[name][0]
    order = np.argsort(np.where(np.isfinite(values), values, -np.inf))[::-1][:5]
    print(f"{name:>14s}   " + ",  ".join(
        f"{BAND_LOW[i]:g}-{BAND_HIGH[i]:g} Hz ({values[i]:.3g})" for i in order))

## The transfer functions behind the cells

A grid summarises each band to one number; this is what those numbers are summarising. The
bands drawn are whichever the chosen metric ranks highest, plus the notebook-7 row ($a = 0$)
for comparison, so a cell that scores well can be checked for whether it looks like a response
at all.

In [ ]:
# --- options -----------------------------------------------------------------
EXAMPLE_METRIC = "split_half" if HAS_NULL else "baseline_snr"
N_EXAMPLES = 4
EXAMPLE_LAGS = (-100.0, 400.0)
EXAMPLE_FIGSIZE = (6.27, 4.4)
# Bands to always include, as (low, high) in Hz -- None drops them.
REFERENCE_BANDS = [(0.0, 10.0), (0.0, 30.0)]
# =============================================================================

values = METRICS[EXAMPLE_METRIC][0]
chosen = list(np.argsort(np.where(np.isfinite(values), values, -np.inf))[::-1][:N_EXAMPLES])
for low, high in REFERENCE_BANDS or []:
    match = np.flatnonzero(np.isclose(BAND_LOW, low) & np.isclose(BAND_HIGH, high))
    if match.size and match[0] not in chosen:
        chosen.append(int(match[0]))

shown = (times >= EXAMPLE_LAGS[0]) & (times <= EXAMPLE_LAGS[1])
rows = -(-len(chosen) // 2)
example_fig, example_axes = plt.subplots(rows, 2, figsize=EXAMPLE_FIGSIZE, sharex=True,
                                         squeeze=False)
for ax, index in zip(example_axes.ravel(), chosen):
    trace = band_reduced(index)
    ax.plot(times[shown], trace[shown], color=INK, lw=1.2, zorder=4, label="real")
    if HAS_NULL:
        ax.plot(times[shown], band_reduced(index, NULL_FITS)[shown], color="#e2431f",
                lw=0.9, alpha=0.8, zorder=3, label="null")
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.set_title(f"{BAND_LOW[index]:g}-{BAND_HIGH[index]:g} Hz   "
                 f"$r$={split_half[index]:.2f}", fontsize=SMALL_FONTSIZE)
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
    ax.yaxis.get_offset_text().set_fontsize(SMALL_FONTSIZE)
    bare(ax)
for ax in example_axes.ravel()[len(chosen):]:
    ax.set_axis_off()
for ax in example_axes[-1]:
    ax.set_xlabel("Lag (ms)")
example_axes[0, 0].legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="upper right")
example_fig.tight_layout()

stem = f"onset_grid_examples_{REFERENCE}"
example_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
example_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"ranked by {EXAMPLE_METRIC}")
for index in chosen:
    print(f"  {BAND_LOW[index]:5.0f}-{BAND_HIGH[index]:<5.0f} Hz  peak {peak[index]:.3g} "
          f"@ {peak_lag[index]:5.0f} ms  split-half r {split_half[index]:+.2f}"
          + (f"  null SNR {null_snr[index]:.2f}" if HAS_NULL else ""))

## Explained EEG power

Everything above reads a cell as an amplitude in arbitrary units, through a display filter whose
passband had to be chosen before the answer was known -- and the choice is not innocent, because
it *is* half of the question. "Which envelope bands carry edges worth modelling" and "which EEG
frequencies do those bands model" are one question in two dimensions, and collapsing the EEG axis
through a fixed filter answers the first by assuming the second.

The run now saves both auto-spectra, so neither has to be assumed. $H = S_{xy}/S_{xx}$ is the
transfer function; $|H|^2 S_{xx}$ is the EEG power the band's feature *predicts* at each
frequency; over $S_{yy}$ it is the fraction of the EEG there that the band accounts for. That
number is bounded, it is comparable across bands and across EEG frequencies, and it is the thing
a model is actually judged on -- unlike a TRF peak, which is in units of EEG per feature and
rewards whichever band has the smallest regressor.

The null run is the floor. One regressor explains some EEG power by chance, the more so the
fewer independent segments there are, and the mispaired fit measures exactly that bias with
everything else held fixed. Biases add, so the excess is a difference rather than a ratio.

**What this can and cannot settle.** It is still a univariate reading: every cell is fitted
alone, and the bands overlap heavily by construction. A wide band beats a narrow one partly
because it has more feature energy, not only because its edges matter more, so "the best cell"
is not "the band to extract". What the grid does support is the nested comparison -- widening
$(a, b)$ to $(a, b')$ changes one thing -- and that is the form the question should be asked in
before any of this becomes a modelling decision.

In [ ]:
# --- options -----------------------------------------------------------------
# The fraction is formed per channel and then averaged over them, not formed from
# channel-averaged spectra: EEG power varies over the scalp by more than the response does, and
# a ratio of averages is dominated by whichever sensors are noisiest.
EXPLAINED_CHUNK = 256          # bands held in memory at once
# =============================================================================


def explained_fraction(path, chunk=EXPLAINED_CHUNK):
    """The fraction of EEG power each band's feature accounts for, `(n_bands, F)`.

    Every array in the file is a weighted sum over trials, and `half_weight` is what each half
    was weighted by -- one in a whole run, less under `--max-stimuli` -- so each is divided by
    it before anything is divided into anything else. The two halves are pooled here; their
    disagreement is what `split_half` above is for.
    """
    with np.load(path) as saved:
        if "half_weight" not in saved:
            raise KeyError(f"{path.name} predates the auto-spectra; re-run get_onset_grid.py")
        weight = saved["half_weight"]
        feature_power = (saved["feature_power"] / weight[:, None, None]).mean(0)   # (B, F)
        eeg_power = (saved["eeg_power"] / weight[:, None, None]).mean(0)           # (F, C)
        fits = saved["fits"]
        fraction = np.empty(feature_power.shape)
        for start in range(0, fits.shape[1], chunk):
            block = (fits[:, start:start + chunk] / weight[:, None, None, None]).mean(0)
            predicted = np.abs(block) ** 2 * feature_power[start:start + chunk, :, None]
            fraction[start:start + chunk] = (predicted / eeg_power).mean(-1)
        return fraction


explained = explained_fraction(GRID_PATH)
explained_null = explained_fraction(NULL_PATH) if HAS_NULL else np.zeros_like(explained)
# Biases add, so the floor comes off rather than dividing: what is left is the EEG power this
# band accounts for over and above what a mispaired regressor of the same shape accounts for.
excess = explained - explained_null

# Mains harmonics are a shared artefact of both signals and would be read as explained power.
eeg_keep = np.abs(frequencies[:, None] - np.arange(MAINS_HZ, frequencies[-1] + MAINS_HZ,
                                                   MAINS_HZ)).min(1) > NOTCH_BANDWIDTH_HZ
eeg_keep &= frequencies > 0

best = np.unravel_index(np.argmax(np.where(eeg_keep, excess, -np.inf)), excess.shape)
print(f"explained fraction: floor (null) median {np.median(explained_null[:, eeg_keep]):.2e}, "
      f"best cell {excess[best]:.2e} above it")
print(f"  {BAND_LOW[best[0]]:g}-{BAND_HIGH[best[0]]:g} Hz envelope band, "
      f"EEG {frequencies[best[1]]:g} Hz")
print(f"\n{'EEG band':>12}  {'best envelope band':>20}  {'excess':>9}  {'floor':>9}")
for low, high in [(0.5, 4), (4, 8), (8, 15), (15, 30), (30, 45)]:
    inside = eeg_keep & (frequencies >= low) & (frequencies <= high)
    per_band = excess[:, inside].mean(1)
    index = int(np.argmax(per_band))
    print(f"{low:5g}-{high:<5g} Hz  {BAND_LOW[index]:8g}-{BAND_HIGH[index]:<8g} Hz  "
          f"{per_band[index]:9.2e}  {explained_null[index, inside].mean():9.2e}")